# Batch Inference for Fine-tuned Models (Vertex AI Custom Job)

This notebook implements batch inference for fine-tuned models using Cloud Vertex AI. We use batch inference because:
1. We want to run inference on the test dataset that the model has not seen during training for evaluation.
2. Real-time inference would be too costly and slow for our dataset size
3. We use Vertex AI aiplatform.CustomJob to have the option to run inference in the local environment or with GPU instances remotely without code changes.

**FLow**
```
GCS (test data + model adapter)
        ↓
Vertex AI Custom Job (GPU)
  └─ infer.py: download → vLLM infer → upload results
        ↓
GCS (results.jsonl)
        ↓
Notebook: download → tracking CSV → Notebook 05 evaluate
```

In [1]:
print("Notebook 04 run")

Notebook 04 run


In [2]:
# setup checked — no SageMaker dependencies needed

## Import Required Libraries

In [3]:
%pip install -U --quiet google-cloud-aiplatform google-cloud-storage python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
import json, os, subprocess, csv
from pathlib import Path
import pandas as pd
from typing import Union, Dict, Optional
from IPython.display import display, HTML
from ipywidgets import widgets

from google.cloud import aiplatform, storage

In [6]:
import os, json
print(os.getcwd())
print(os.path.exists("gcs_config.json"))

d:\Internship-Biwoco\Fine-tune\sample-for-multi-modal-document-to-json-with-vertex-ai
True


In [7]:
import os, json
from google.cloud import aiplatform, storage

with open("01_gcs_config.json") as f:
    gcs_cfg = json.load(f)

PROJECT_ID         = gcs_cfg["project_id"]
default_bucket_name = gcs_cfg["bucket_name"]
region             = gcs_cfg["region"]
dataset_gcs_prefix = gcs_cfg["gcs_output_prefix"]
gcs_root_uri       = f"gs://{default_bucket_name}"
dataset_gcs_uri    = f"{gcs_root_uri}/{dataset_gcs_prefix}"
gcs_model_dir      = f"{gcs_root_uri}/{gcs_cfg['gcs_model_dir']}"
gcs_results        = f"{gcs_root_uri}/inference_results"

aiplatform.init(project=PROJECT_ID, location=region)
print(f"Project: {PROJECT_ID} | Region: {region}")
print(f"Dataset: {dataset_gcs_uri}")
print(f"Model dir: {gcs_model_dir}")

Project: first-orc-chien | Region: asia-southeast1
Dataset: gs://electric-bill-dataset-gcs/data/swift_dataset
Model dir: gs://electric-bill-dataset-gcs/output/model


## Retrieve model artifact location from fine-tuning

In [8]:
from dataclasses import dataclass

@dataclass
class ModelConfig:
    model_type: str
    model_id: str

    def training_job_prefix(self, dataset_prefix: str) -> str:
        name = self.model_id.replace("/", "-").replace(".", "-")
        return f"train-{name}"

base_model_config = ModelConfig(
    model_type="qwen2_5_vl",
    model_id="Qwen/Qwen2.5-VL-7B-Instruct",
)
print(f"Base model: {base_model_config.model_id}")

Base model: Qwen/Qwen2.5-VL-7B-Instruct


In [9]:
training_job_name_prefix = base_model_config.training_job_prefix(dataset_gcs_prefix)
print(f"Fine-tuning name prefix: {training_job_name_prefix}")

Fine-tuning name prefix: train-Qwen-Qwen2-5-VL-7B-Instruct


In [10]:
display(HTML(f"""
<div style="border: 2px solid #006CE0; 
    padding: 10px; 
    border-radius: 5px; 
    max-width: 100%;
    background: #f0fbff;">
    <b>Note:</b> Skip the next 4 cells below if you want to run inference using <b>{base_model_config.model_id}</b> from HuggingFace Hub. 
    <br>Run the cells below if you want to use a model that you have fine-tuned.
</div>
"""))

In [11]:
!gcloud storage buckets add-iam-policy-binding {gcs_root_uri} \
  --member="user:ngocchiien23@gmail.com" \
  --role="roles/storage.objectViewer"

bindings:
- members:
  - projectEditor:first-orc-chien
  - projectOwner:first-orc-chien
  role: roles/storage.legacyBucketOwner
- members:
  - projectViewer:first-orc-chien
  role: roles/storage.legacyBucketReader
- members:
  - projectEditor:first-orc-chien
  - projectOwner:first-orc-chien
  role: roles/storage.legacyObjectOwner
- members:
  - projectViewer:first-orc-chien
  role: roles/storage.legacyObjectReader
- members:
  - serviceAccount:73397202200-compute@developer.gserviceaccount.com
  - user:ngocchiien23@gmail.com
  role: roles/storage.objectAdmin
- members:
  - serviceAccount:custom-online-prediction@a00ba840168c92a5f-tp.iam.gserviceaccount.com
  - serviceAccount:custom-online-prediction@wbb7125df6ba5f242-tp.iam.gserviceaccount.com
  - user:ngocchiien23@gmail.com
  role: roles/storage.objectViewer
etag: CAw=
kind: storage#policy
resourceId: projects/_/buckets/electric-bill-dataset-gcs
version: 1


In [12]:
print(gcs_model_dir)

gs://electric-bill-dataset-gcs/output/model


In [13]:
!where gsutil

D:\Google\Cloud SDK\google-cloud-sdk\bin\gsutil
D:\Google\Cloud SDK\google-cloud-sdk\bin\gsutil.cmd


In [14]:
import subprocess, re

result = subprocess.run(
    f'gsutil ls "{gcs_model_dir}/"',
    capture_output=True, text=True, shell=True
)
lines = [l.strip() for l in result.stdout.splitlines() if l.strip()]
df_models = __import__("pandas").DataFrame({"Key": lines})
print(f"Found {len(df_models)} objects in {gcs_model_dir}:")
display(df_models)

Found 1 objects in gs://electric-bill-dataset-gcs/output/model:


,Key
0,gs://electric-bill-dataset-gcs/output/model/v0...


In [15]:
which_model_to_pick = 0 # use first model from list by default. Change to use a different model from list above.

In [16]:
# Set up the URI from which we will download the model
model_output_url = df_models['Key'].iloc[which_model_to_pick]
print(f"Selected model: {model_output_url}")

Selected model: gs://electric-bill-dataset-gcs/output/model/v0-20260623-161620/


<div style="border: 2px solid #006CE0; 
    padding: 10px; 
    border-radius: 5px; 
    max-width: 100%;
    background: #f0fbff;">
    Continue below for inference with base model or fine-tuned model.
</div>

In [17]:
try:
    model_config = ModelConfig(
        # Replace with model type and model id of the base model.
        model_type=base_model_config.model_type,
        model_id=model_output_url
    )
    prefix = model_config.model_id.replace("/","-").replace(".","-")
    
    # prefix = model_suffix_s3.split("/")[0]
    # print("✅ Configured fine-tuned model id.")
    
except NameError:
    # not using fine-tuned model
    model_config = base_model_config
    prefix = model_config.model_id.replace("/","-").replace(".","-")
    print("✅ Using base model for inference.")

In [18]:
print(f"Model for inference: {model_config.model_id}")

Model for inference: gs://electric-bill-dataset-gcs/output/model/v0-20260623-161620/


## Configure Job for Batch Inference

Lets define the Vertex AI Custom Job configuration

In [19]:
# = config matching notebook 03
CONTAINER_URI     = "asia-docker.pkg.dev/vertex-ai/training/pytorch-gpu.2-3.py310:latest"
MACHINE_TYPE      = "a2-highgpu-1g"
ACCELERATOR_TYPE  = "NVIDIA_TESLA_A100"
ACCELERATOR_COUNT = 1

Define the dependencies that are required for inference.

In [20]:
# phải khớp với versions trong train.py
requirements = [
    "ms-swift==3.5.3",
    "transformers==4.52.4",
    "qwen_vl_utils==0.0.11",
    "accelerate",
    "hf_transfer",
]

### Environment Variables Configuration

We set specific environment variables because:
1. Memory usage needs to be optimized for GPUs
2. Image processing has size constraints
3. We want faster downloads from Hugging Face
4. Resource limits need to be carefully managed

In [21]:
# defines the environment variables for the training
env_variables ={
    "SIZE_FACTOR": json.dumps(8), # can be increase but requires more GPU memory
    "MAX_PIXELS": json.dumps(1048576), # can be increase but requires more GPU memory
    "USE_HF_TRANSFER": json.dumps(1),
    "HF_HUB_ENABLE_HF_TRANSFER": json.dumps(1),
    # "HF_TOKEN": "xxxxxxxx",
}


In [22]:
from datetime import datetime
timestamp       = datetime.now().strftime("%Y%m%d%H%M%S")
job_name_prefix = f"infer-{prefix}-{timestamp}"[:60]
print(f"Job name: {job_name_prefix}")

Job name: infer-gs:--electric-bill-dataset-gcs-output-model-v0-2026062


### Constrained Decoding

Constrained decoding controls a language model's next-token prediction process by limiting which tokens it can generate to only those that satisfy specific rules or formats. During the normal generation process, a language model assigns probabilities to all possible next tokens. With constrained decoding the set of next tokens is limited to only tokens that satisfy the required structure. For example with JSON constrained decoding the model can only select tokens that create a valid JSON syntax. 

Below you can configure constrained decoding for the batch inference:
1. Set it to `None` to run batch inference without any constrained decoding.
2. If you have a JSON schema file in your dataset you can set `guided_decoding` to the path of that JSON schema file inside your dataset, for example `guided_decoding = "groundtruth_schema.json"`. You can reference the [02_create_custom_dataset_swift.ipynb](02_create_custom_dataset_swift.ipynb) notebook on how to create a JSON schema file. 
3. You can also set `guided_decoding` to a dict sturctured output parameter from the [vLLM documentation](https://docs.vllm.ai/en/latest/features/structured_outputs.html), for example `guided_decoding = {"guided_choice": ["positive", "negative"]}`

In [23]:
guided_decoding = None # 1. default no constrained decoding

# guided_decoding = "groundtruth_schema.json" # 2. use a JSON schema inside dataset

# 3. Below is an example on how to configure structure output in accordance to the vLLM documentation
# from pydantic import BaseModel

# class Invoice(BaseModel):
#     purpose: str
#     amount: int

# json_schema = Invoice.model_json_schema()
# guided_decoding = {"guided_json": json_schema}

## Batch Inference Function

In [24]:
def batch_inference(model_id, model_type, dataset_gcs,
                    test_data_path="conversations_test_swift_format.json",
                    guided_decoding=None):

    run_results_gcs = f"{gcs_results}/{job_name_prefix}"

    job = aiplatform.CustomJob.from_local_script(
        display_name=job_name_prefix,
        script_path="infer.py",
        container_uri=CONTAINER_URI,
        staging_bucket=gcs_results, 
        requirements=requirements,
        machine_type=MACHINE_TYPE,
        accelerator_type=ACCELERATOR_TYPE,
        accelerator_count=ACCELERATOR_COUNT,
        environment_variables=env_variables,
        args=[
            "--model_id",       model_id,
            "--model_type",     model_type,
            "--dataset_gcs",    dataset_gcs,
            "--test_data_path", test_data_path,
            "--results_gcs",    run_results_gcs,
            "--guided_decoding", json.dumps(guided_decoding),
        ],
    )
    job.run(sync=True)
    print(f"✅ Job done: {run_results_gcs}")
    return run_results_gcs

## Run Batch Inference

In [25]:
inference_kwargs = {
    "model_id":        model_config.model_id,
    "model_type":      model_config.model_type,
    "dataset_gcs":      dataset_gcs_uri,   
    "test_data_path":  "conversations_test_swift_format.json",
    "guided_decoding": guided_decoding
}

In [26]:
print(f"View your job here: https://console.cloud.google.com/vertex-ai/training/custom-jobs?project={PROJECT_ID}")
# inference_output_url = batch_inference(**inference_kwargs)

View your job here: https://console.cloud.google.com/vertex-ai/training/custom-jobs?project=first-orc-chien


In [27]:
inference_output_url = "gs://electric-bill-dataset-gcs/inference_results/infer-Qwen-Qwen2-5-VL-3B-Instruct-20260706102900"
# job_name_prefix = "infer-Qwen-Qwen2-5-VL-3B-Instruct-20260706102900"

In [28]:
print(f"📊 Inference results: {inference_output_url}/results.jsonl")

📊 Inference results: gs://electric-bill-dataset-gcs/inference_results/infer-Qwen-Qwen2-5-VL-3B-Instruct-20260706102900/results.jsonl


### Download inference results

### Track Inference Results

We track inference results in a CSV file for evaluation of different models later.
1. We need to maintain a history of all inference runs
2. We want to associate results with specific models
3. We need to easily locate model outputs later
4. CSV format enables easy tracking

In [29]:

# copy link inference result
!gsutil cat {inference_output_url}/results.jsonl > results.jsonl
# !gsutil cat gs://electric-bill-dataset-gcs/inference_results/infer-Qwen-Qwen2-5-VL-3B-Instruct-20260706102900/results.jsonl > results.jsonl

In [30]:
with open("results.jsonl", "r", encoding="utf-8") as f:
    print(f.read(1000))

{"response": "```json\n{\n  \"account_holder_name\": \"Regina Williamson\",\n  \"account_number\": \"9215687\",\n  \"amount_due\": \"$50.73\",\n  \"average_daily_usage_kwh\": \"1.219\",\n  \"bill_due_date\": \"17 March 2025\",\n  \"bill_issue_date\": \"25 February 2025\",\n  \"billing_period_end\": \"17 Feb 25\",\n  \"billing_period_start\": \"19 Jan 25\",\n  \"document_type\": \"Electricity Account\",\n  \"nmi\": \"61993993987\",\n  \"provider_abn\": \"86 601 199 151\",\n  \"provider_name\": \"SUMO\",\n  \"service_address_postcode\": \"3064\",\n  \"service_address_state\": \"VIC\",\n  \"service_address_street\": \"Apt. 059 68 Dylan Dip\",\n  \"service_address_suburb\": \"Roxburgh Park\",\n  \"supply_charge_days\": \"29 days\",\n  \"tax_invoice_number\": \"6574958\",\n  \"total_electricity_kwh\": \"24.835 kWh\",\n  \"total_excl_gst\": \"$50.73\",\n  \"total_gst\": \"$4.61\",\n  \"total_incl_gst\": \"$50.73\",\n  \"usage_discount_amount\": \"-$3.36\",\n  \"usage_discount_percent\": \"-3

In [31]:
!gsutil cp {inference_output_url}/results.jsonl .
# !gsutil cp "gs://electric-bill-dataset-gcs/inference_results/infer-Qwen-Qwen2-5-VL-3B-Instruct-2026070610290/results.jsonl" .

Copying gs://electric-bill-dataset-gcs/inference_results/infer-Qwen-Qwen2-5-VL-3B-Instruct-20260706102900/results.jsonl...
/ [0 files][    0.0 B/615.0 KiB]                                                
-
- [1 files][615.0 KiB/615.0 KiB]                                                

Operation completed over 1 objects/615.0 KiB.                                    


## Next step
* Continue with the [05_evaluate_model.ipynb](./05_evaluate_model.ipynb) notebook to evaluate the models performance and compare it to other models. 

## Inference performance report (time / tokens / RAM-GPU)

Read-only cells: pulls from Cloud Logging, Cloud Monitoring, and `results.jsonl`. Does not modify `infer.py` or any cell above.


In [32]:
%pip install -U --quiet google-cloud-logging google-cloud-monitoring transformers sentencepiece torchvision


Note: you may need to restart the kernel to use updated packages.


ERROR: Exception:
Traceback (most recent call last):
  File "d:\Anaconda\envs\ocr\Lib\site-packages\pip\_vendor\urllib3\response.py", line 897, in _error_catcher
    yield
  File "d:\Anaconda\envs\ocr\Lib\site-packages\pip\_vendor\urllib3\response.py", line 1022, in _raw_read
    data = self._fp_read(amt, read1=read1) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\envs\ocr\Lib\site-packages\pip\_vendor\urllib3\response.py", line 1005, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\envs\ocr\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 104, in read
    self.__buf.write(data)
  File "d:\Anaconda\envs\ocr\Lib\tempfile.py", line 483, in func_wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 28] No space left on device

The above exception was the direct cause of the following exception:

Traceback (most rec

In [33]:
from pathlib import Path
import pandas as pd
from utils.inference_report import (
    find_latest_inference_job, download_job_results,
    fetch_job_log_lines, parse_phase_timings, parse_per_sample_timings,
    read_results, load_tokenizer, compute_output_token_stats, compute_input_token_stats,
    fetch_gpu_cpu_stats, build_report_row, append_to_report_csv, find_job_by_id
)

REPORT_CSV = Path("inference_performance_report.csv")
RESULTS_JSONL = Path("results.jsonl")
LOCAL_DATASET_DIR = Path("dataset_local")


In [34]:
#Find job and download results.jsonl về local
job = find_latest_inference_job(PROJECT_ID, region, gcs_results)
job_id = job.resource_name.split("/")[-1]
total_job_seconds = (job.end_time - job.start_time).total_seconds() if job.start_time and job.end_time else None
download_job_results(job.display_name, gcs_results, RESULTS_JSONL)

print(f"job: {job.display_name}")
print(f"job_id: {job_id}")
print(f"total_job_seconds: {total_job_seconds}")

job: infer-gs:--electric-bill-dataset-gcs-output-model--202606241
job_id: 8170502516962754560
total_job_seconds: 2805.0


In [35]:
#  timing, from Cloud Logging
log_lines = fetch_job_log_lines(job_id, PROJECT_ID, job.start_time, job.end_time)
print(f"fetched {len(log_lines)} log lines")

phase_timings = parse_phase_timings(log_lines)
sample_timings = parse_per_sample_timings(log_lines)
for name, value in {**phase_timings, **sample_timings}.items():
    print(f"{name}: {value}")


fetched 7270 log lines
setup_to_checkpoint_seconds: 33.800447
checkpoint_to_infer_done_seconds: 2627.599868
infer_done_to_upload_seconds: 2.400147
sample_time_seconds_min: 13.0
sample_time_seconds_max: 17.0
sample_time_seconds_mean: 15.323170731707316
sample_time_seconds_std: 0.7803687028027833
total_samples_timed: 164


In [36]:
results = read_results(RESULTS_JSONL)
tokenizer = load_tokenizer(model_config.model_id, base_model_config.model_id)

output_token_stats = compute_output_token_stats(results, tokenizer)
input_token_stats = compute_input_token_stats(results, base_model_config.model_id, dataset_gcs_uri, LOCAL_DATASET_DIR)

tokens_per_sec = None
if phase_timings.get("checkpoint_to_infer_done_seconds"):
    tokens_per_sec = output_token_stats["total"] / phase_timings["checkpoint_to_infer_done_seconds"]

print("output tokens:", output_token_stats)
print("input tokens:", input_token_stats)
print(f"tokens/sec: {tokens_per_sec:.1f}" if tokens_per_sec else "tokens/sec: N/A")

torchvision not found — falling back to text-only token count (install torchvision for accurate stats)
output tokens: {'min': 292.0, 'max': 343.0, 'mean': 313.5030303030303, 'std': 12.049707272905763, 'total': 51728.0, 'n_samples': 165}
input tokens: {'min': 425.0, 'max': 478.0, 'mean': 447.95757575757574, 'std': 12.663166110465049, 'text_only_fallback': True}
tokens/sec: 19.7


In [37]:
# Step 4: GPU/CPU utilization, from Cloud Monitoring (best-effort)
resource_stats = fetch_gpu_cpu_stats(job_id, PROJECT_ID, job.start_time, job.end_time)
for name, stats in resource_stats.items():
    print(f"{name}: {stats}")


cpu_utilization: {'min': 0.02201470860301665, 'max': 0.3242231933160786, 'mean': 0.11911179804932016, 'std': 0.04548765315430431}
gpu_utilization: {'min': 0.0, 'max': 0.44, 'mean': 0.3595652173913043, 'std': 0.12379781064544282}
gpu_memory: {'min': 0.0151580810546875, 'max': 0.40848846435546876, 'mean': 0.3707551022793382, 'std': 0.11222838284764798}


In [39]:
# Step 5: assemble one report row, append to CSV history
report_row = build_report_row(
    job, job_id, model_config.model_id, total_job_seconds, tokens_per_sec,
    phase_timings, sample_timings, output_token_stats, input_token_stats, resource_stats,
)
history_df = append_to_report_csv(report_row, REPORT_CSV)

display(pd.DataFrame([report_row]).T)
print(f"saved to {REPORT_CSV.resolve()} ({len(history_df)} runs total)")


,0
run_timestamp,2026-07-08T05:17:01.980712+00:00
job_display_name,infer-gs:--electric-bill-dataset-gcs-output-mo...
job_id,8170502516962754560
model_id,gs://electric-bill-dataset-gcs/output/model/v0...
total_job_seconds,2805.0
tokens_per_sec,19.686407
setup_to_checkpoint_seconds,33.800447
checkpoint_to_infer_done_seconds,2627.599868
infer_done_to_upload_seconds,2.400147
sample_time_seconds_min,13.0


saved to D:\Internship-Biwoco\Fine-tune\sample-for-multi-modal-document-to-json-with-vertex-ai\inference_performance_report.csv (1 runs total)
